Field validataion in Pydantic
We validate the field and do some transformation

In [ ]:
from pydantic import Field, BaseModel, EmailStr, AnyUrl, field_validator
from typing import List, Dict, Optional, Annotated

class Patient(BaseModel):
    name: str
    email: EmailStr
    age: int
    weight: float
    married: bool
    allergies: List[str] = None
    contact_details: Dict[str, str]

    @field_validator('email')
    @classmethod
    def email_validator(cls, value):
        valid_domains = ['hdfc.com', 'icici.com']
        #abc@gmail.com

        domain_name = value.split('@')[-1]
        if domain_name not in valid_domains:
            raise ValueError('Not a valid domain')

        return value

def update_patient_data(pat_info: Patient):
    print(pat_info.name)
    print(pat_info.email)
    print(pat_info.age)
    print(pat_info.weight) 
    print(pat_info.allergies)
    print(pat_info.married)
    print('Updated...')


patient_info = {'name':'Mustafa','email':'abc@hdfc.com', 'age': 21, 'weight':75.2, 'married':True, 'contact_details':{'email':'abc@gmail.com', 'phone':'123145154'}}

patient1 = Patient(**patient_info)

update_patient_data(patient1)
    

Mustafa
abc@hdfc.com
21
75.2
None
True
Updated...


# Field validator 
mode = after (default), before 

before : before type conrection, it will check. If both the datatype is different then it will return an error.

age : int
if we pass '45', then it will throwing error when field validator use before
if we use after, then it will passed through, str convert to int

In [12]:
from pydantic import Field, BaseModel, EmailStr, AnyUrl, field_validator
from typing import List, Dict, Optional, Annotated

class Patient(BaseModel):
    name: str
    email: EmailStr
    age: int
    weight: float
    married: bool
    allergies: List[str] = None
    contact_details: Dict[str, str]

    @field_validator('email')
    @classmethod
    def email_validator(cls, value):
        valid_domains = ['hdfc.com', 'icici.com']
        #abc@gmail.com

        domain_name = value.split('@')[-1]
        if domain_name not in valid_domains:
            raise ValueError('Not a valid domain')

        return value
    # Data transformation done here, upper case the name 

    @field_validator('name', mode='after')
    @classmethod
    def transform_name(cls, value):
        return value.upper()

    @field_validator('age', mode='before')
    @classmethod
    def validate(cls, value):
        if 0 < value < 100:
            return value
        else:
            raise ValueError('Age should be in between 0 and 100')

def update_patient_data(pat_info: Patient):
    print(pat_info.name)
    print(pat_info.email)
    print(pat_info.age)
    print(pat_info.weight) 
    print(pat_info.allergies)
    print(pat_info.married)
    print('Updated...')


patient_info = {'name':'Mustafa','email':'abc@hdfc.com', 'age': '21', 'weight':75.2, 'married':True, 'contact_details':{'email':'abc@gmail.com', 'phone':'123145154'}}

patient1 = Patient(**patient_info) #validation -> type conercion

update_patient_data(patient1)
    

TypeError: '<' not supported between instances of 'int' and 'str'

#Model_validator

 Dependent validation check, means one field value passed through then next field will gives the result. So here on field depends on another.

example: Age > 60, then emergency contact number required always

In [20]:
from pydantic import Field, BaseModel, EmailStr, AnyUrl, field_validator,model_validator
from typing import List, Dict, Optional, Annotated

class Patient(BaseModel):
    name: str
    email: EmailStr
    age: int
    weight: float
    married: bool
    allergies: List[str] = None
    contact_details: Dict[str, str]

    # Model validator 
    @model_validator(mode='after')
    def validate_emergency_contact(cls, model):
        if model.age > 60 and 'emergency' not in model.contact_details:
            raise ValueError('Patient older than 60 must have emergency contact')
        return model

def update_patient_data(pat_info: Patient):
    print(pat_info.name)
    print(pat_info.email)
    print(pat_info.age)
    print(pat_info.weight) 
    print(pat_info.allergies)
    print(pat_info.married)
    print('Updated...')

patient_info = {'name':'Mustafa','email':'abc@hdfc.com', 'age': '62', 'weight':75.2, 'married':True, 'contact_details':{'email':'abc@gmail.com', 'phone':'123145154', 'emergency':'191'}}

patient1 = Patient(**patient_info) #validation -> type conercion

update_patient_data(patient1)

Mustafa
abc@hdfc.com
62
75.2
None
True
Updated...


C:\Users\smsir\AppData\Local\Temp\ipykernel_37020\3973589965.py:14: PydanticDeprecatedSince212: Using `@model_validator` with mode='after' on a classmethod is deprecated. Instead, use an instance method. See the documentation at https://docs.pydantic.dev/2.13/concepts/validators/#model-after-validator. Deprecated in Pydantic V2.12 to be removed in V3.0.
  @model_validator(mode='after')
